# X2 · Triangulación de los tres ejemplaresComparar que **las tres actas de cada mesa digan lo mismo en el papel**.Es la señal que la aritmética no puede ver: mover votos de un candidato a otrodeja el total intacto, así que ninguna suma lo delata. Sólo comparar ejemplareslo revela.## Dos cosas que hay que saber antes de leer los resultados### 1 · El voto mayoritario 2-vs-1 NO sirve aquíParece natural que si dos ejemplares coinciden y uno difiere, el discrepante seael sospechoso. **Es falso en este corpus.** DELEGADOS y TRANSMISIÓN son ambos**binarizados de 1 bit**; comparten modo de fallo y se equivocan igual, formandomayorías falsas. Medido en Turbo:| Mesa | CLAVEROS | DELEGADOS | TRANSMISIÓN | Oficial ||---|---|---|---|---|| 6 | **121** | 21 | 21 | **121** || 15 | **119** | 19 | 19 | **119** |Los dos binarizados pierden el dígito de **centenas**; el discrepante era el queacertaba. Por eso el árbitro es el **escrutinio oficial**, independiente del papely verificado por SHA-256.### 2 · La confianza clasifica, no descartaUna casilla retocada o emborronada **hace dudar al modelo precisamente por estaralterada**. Filtrar por confianza alta eliminaría justo lo que se busca. Por esolas divergencias se ordenan por confianza en vez de filtrarse:- **ALTA** (≥0,90) — el modelo estaba seguro: una divergencia aquí es sólida.- **BAJA** (<0,70) — puede ser ruido del OCR **o** una casilla difícil por estar  alterada. Hay que mirarla, no tirarla.

## 0 · Configuración

In [ ]:
from pathlib import Path
import sys, time, torch

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "e14").is_dir())
sys.path.insert(0, str(RAIZ)); sys.path.insert(0, str(RAIZ / "e14" / "extraccion"))

CLAVEROS    = RAIZ / "data/segunda_vuelta/e14_pdfs_claveros"
DELEGADOS   = RAIZ / "data/segunda_vuelta/e14_pdfs_2v"
TRANSMISION = RAIZ / "data/segunda_vuelta/e14_pdfs_2v_t"
MMV         = RAIZ / "data/manifests/MMV_2V/MMV_Presidente2V_2026"
MODELO      = RAIZ / "models/digitnet_2v_relleno.pt"
SALIDA      = RAIZ / "data/segunda_vuelta/_triangulacion/tri_2v"

N_MESAS  = 3000     # None = las 118.337 (≈8-10 h). Empezá con 3.000 (~15 min).
CONF_MIN = 0.90     # umbral para etiquetar una divergencia como ALTA
SOLO_CANDIDATOS = False   # False = las 6 casillas; True = sólo los dos candidatos

if torch.cuda.is_available():
    libre, total = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name(0)}   VRAM libre {libre/1e9:.1f}/{total/1e9:.1f} GB")
print("modelo:", MODELO.name, "|", "existe" if MODELO.exists() else "NO EXISTE")
print(f"mesas a triangular: {N_MESAS or 'todas'}   (lee 3 actas por mesa)")

## 1 · Validar el OCR en los tres ejemplares**Imprescindible antes de interpretar nada.** El modelo se entrenó sólo conCLAVEROS; si lee peor los binarizados, aportarán divergencias falsas enproporción a su error.Referencia medida sobre 60 mesas: CLAVEROS 98,0 % · DELEGADOS 97,3 % ·TRANSMISIÓN 95,3 %.

In [ ]:
from e14.comparacion.triangular import validar_ocr
from e14.ocr.clasificador_color import cargar
from e14.oficial import mmv
from collections import defaultdict

f = mmv.localizar(MMV)
esc = mmv.cargar_escrutinio(f["escrutinio"])
tot = defaultdict(int)
for v in esc.values():
    for c, n in v.items():
        tot[c] += n
por_cod = {c[1]: c for c, v in tot.items() if mmv.es_candidato(c) and v > 0}

red, dev = cargar(str(MODELO), "cuda" if torch.cuda.is_available() else "cpu")
dirs = {"CLAVEROS": str(CLAVEROS), "DELEGADOS": str(DELEGADOS), "TRANSMISION": str(TRANSMISION)}

v = validar_ocr(dirs, esc, por_cod, red, dev, n=80)
for e, (acc, n) in v.items():
    print(f"  {e:12s} {acc:.1%}  ({n:,} casillas)")
peor = min(a for a, _ in v.values()); mejor = max(a for a, _ in v.values())
print(f"\\ndiferencia entre el mejor y el peor ejemplar: {100*(mejor-peor):.1f} puntos")
if mejor - peor > 0.05:
    print("! Un ejemplar se lee bastante peor: sus divergencias serán en buena")
    print("  parte fallos de lectura, no del papel. Tenelo presente al leer el CSV.")

## 2 · TriangularLee las tres actas de cada mesa. ~0,3 s por mesa (3 lecturas), VRAM plana.

In [ ]:
from e14.comparacion.triangular import triangular

t0 = time.time()
filas = triangular(str(CLAVEROS), str(DELEGADOS), str(TRANSMISION), str(MMV),
                   str(MODELO), str(SALIDA), limite=N_MESAS, conf_min=CONF_MIN,
                   solo_candidatos=SOLO_CANDIDATOS, validar=False,
                   dev="cuda" if torch.cuda.is_available() else "cpu")
print(f"\\nterminado en {(time.time()-t0)/60:.1f} min")

## 3 · Lo que merece revisión humana`UN_EJEMPLAR_SE_APARTA` con confianza **ALTA**: un ejemplar dice algo distintodel escrutinio oficial y de los otros dos, y el modelo lo leyó seguro.Todo lo demás (varios se apartan, ninguno coincide, confianza baja) esmayoritariamente fallo de OCR.

In [ ]:
import pandas as pd

df = pd.read_csv(f"{SALIDA}.csv")
print("divergencias por tipo y confianza:")
print(pd.crosstab(df.tipo, df.confianza).to_string())

rev = df[(df.tipo == "UN_EJEMPLAR_SE_APARTA") & (df.confianza == "ALTA")].copy()
print(f"\\n=== {len(rev)} casillas donde UN ejemplar se aparta, leídas con ALTA confianza ===")
if len(rev):
    print(rev.sospechoso.value_counts().to_string())
    cols = ["dep","muni","zona","puesto","mesa","casilla",
            "CLAVEROS","DELEGADOS","TRANSMISION","oficial","sospechoso","conf_min"]
    print()
    print(rev[cols].head(30).to_string(index=False))
else:
    print("ninguna: en esta muestra, todas las divergencias son de confianza media/baja,")
    print("lo que apunta a fallos de lectura y no a discrepancias del papel.")

### 3.1 · ¿Se aparta siempre el mismo ejemplar?Si un ejemplar concreto acumula las divergencias, lo más probable es que se leapeor —no que esté alterado—. Compará esto con la validación de la sección 1.

In [ ]:
todas_div = df[df.tipo.isin(["UN_EJEMPLAR_SE_APARTA", "VARIOS_SE_APARTAN"])]
print("veces que cada ejemplar se aparta del oficial:")
from collections import Counter
c = Counter()
for s in todas_div.sospechoso.dropna():
    for e in str(s).split("|"):
        if e: c[e] += 1
for e, n in c.most_common():
    print(f"  {e:12s} {n:,}")
print("\\nSi el reparto es muy desigual, lo esperable es que refleje la calidad de")
print("lectura de cada ejemplar (sección 1), no una alteración del papel.")

## 4 · Evidencia visual**No concluyas nada sin mirar el papel.** Se pintan las tres casillas de cadacaso, una debajo de otra.

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
import posiciones_2v as P
from e14.comparacion.triangular import rutas_de_mesa

CUANTAS = 5
casos = rev if len(rev) else df[df.tipo == "UN_EJEMPLAR_SE_APARTA"].head(CUANTAS)

for _, r in casos.head(CUANTAS).iterrows():
    k = (int(r.dep), int(r.muni), int(r.zona), str(r.puesto).zfill(2), int(r.mesa))
    rutas = rutas_de_mesa(k, dirs)
    tiras = []
    for nom in ("CLAVEROS", "DELEGADOS", "TRANSMISION"):
        if nom not in rutas:
            continue
        celda = P.recortar_celdas(rutas[nom], color=True)[r.casilla]
        celda = cv2.resize(celda, (430, 95))
        if celda.ndim == 2:
            celda = cv2.cvtColor(celda, cv2.COLOR_GRAY2BGR)
        lab = np.full((95, 260, 3), 255, np.uint8)
        marca = " <-- se aparta" if nom in str(r.sospechoso).split("|") else ""
        cv2.putText(lab, nom, (3, 34), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
        cv2.putText(lab, f"lee {r[nom]}{marca}", (3, 66), cv2.FONT_HERSHEY_SIMPLEX,
                    0.42, (200, 0, 0) if marca else (0, 0, 0), 1)
        tiras.append(np.hstack([lab, celda]))
    if tiras:
        plt.figure(figsize=(11, 1.1 * len(tiras)))
        plt.imshow(cv2.cvtColor(np.vstack(tiras), cv2.COLOR_BGR2RGB)); plt.axis("off")
        plt.title(f"{r.dep}/{r.muni}/{r.zona}/{r.puesto}/{r.mesa} · {r.casilla} · "
                  f"oficial={r.oficial} · conf={r.conf_min}", fontsize=9)
        plt.show()

## 5 · Caso de referencia: Turbo (Antioquia)Las 4 mesas registradas en la documentación como "enmendaduras reportadas".Sirven de control: **ya se comprobó que sus divergencias son fallos de OCR**, nodel papel — los binarizados pierden el dígito de centenas y el oficial confirmaque CLAVEROS acertaba.Si tu corrida a escala saca casos con un perfil distinto (confianza ALTA, y elejemplar que se aparta no es sistemáticamente el mismo), esos sí son nuevos.

In [ ]:
turbo = ["01/280/03/01/002", "01/280/03/01/003", "01/280/03/01/006", "01/280/03/01/015"]
_ = triangular(str(CLAVEROS), str(DELEGADOS), str(TRANSMISION), str(MMV), str(MODELO),
               str(SALIDA) + "_turbo", conf_min=CONF_MIN, solo_candidatos=False,
               validar=False, mesas=turbo,
               dev="cuda" if torch.cuda.is_available() else "cpu")

t = pd.read_csv(f"{SALIDA}_turbo.csv")
cols = ["mesa","casilla","CLAVEROS","DELEGADOS","TRANSMISION","oficial","sospechoso","confianza"]
print()
print(t[cols].to_string(index=False))

## 6 · Qué reportarAvisame con:1. **Cuántas** casillas salen `UN_EJEMPLAR_SE_APARTA` con confianza **ALTA**.2. **Qué ejemplar** se aparta más (sección 3.1) y si eso encaja con su calidad de   lectura (sección 1).3. **Si la evidencia visual confirma** alguno de los casos.Con eso decidimos si hay algo que llevar al informe consolidado o si, como enTurbo, todo se explica por la lectura.⚠️ **Una divergencia no es fraude.** Puede ser un error del jurado al copiar elacta entre ejemplares — que es justamente lo que se espera encontrar en unproceso manual. El juicio es humano.